# Checkpoint 13 — Final evaluation of the selected approach

**Starting point:** notebook 12 selected logistic regression using validation Brier/simplicity rules. Gradient boosting had higher validation AP but did not win that predefined rule. We now evaluate the selected configuration against the constant; we do not try every candidate on test or switch winners afterwards.

Refit on mature training+validation shipments using only reports available at test start. Test outcomes are judged retrospectively at the analysis snapshot cutoff. Exclude the last six-hour horizon plus 48-hour allowance where follow-up is too short. All assumptions remain explicit; this is synthetic-data evidence, not a launch decision.

Independent execution rebuilds development selection deterministically from source JSON before accessing test labels. Where a saved selection exists, assert the same choice and input hashes. Earlier output is used as a contract for the next step.


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/dispatch_risk/contracts.py').is_file())
# Import the repository source even when editable-install paths are unavailable.
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'personal' / 'notebooks' / 'support'))
import workflow as wf
import pandas as pd
import numpy as np
from IPython.display import display
import json
saved_path = ROOT / 'personal' / 'outputs' / 'notebook_results' / "validation_selection.json"
saved = json.loads(saved_path.read_text()) if saved_path.exists() else None
selection,model,baseline,development,test,p,base_p = wf.test_run()
if saved is not None:
    assert saved["selected_model"] == selection["selected_model"]
    assert saved["split"] == selection["split"]
result = {"selected_model":selection["selected_model"], "selection_rule":selection["rule"],
          "split":selection["split"], "development_rows":len(development),
          "test":wf.metrics(test.label,p), "constant_test":wf.metrics(test.label,base_p)}
display(pd.DataFrame([{"method":selection["selected_model"],**result["test"]}, {"method":"constant",**result["constant_test"]}]))
print(json.dumps(result,indent=2))


                method  rows  ...  true_negative  false_negative
0  logistic_regression   309  ...            284               1
1             constant   309  ...            284              25

[2 rows x 12 columns]
{
  "selected_model": "logistic_regression",
  "selection_rule": "Improve validation AP and Brier over constant; choose simplest within 0.002 Brier of best eligible; otherwise constant",
  "split": {
    "validation_start": "2026-02-15T08:00:00+00:00",
    "test_start": "2026-03-02T08:00:00+00:00",
    "observation_cutoff": "2026-03-17T11:00:00+00:00",
    "grace_hours": 48,
    "lookback_hours": 3,
    "cohort_rule": "60/20/20 by shipment first decision time; ties assigned by time",
    "completeness_assumption": "Reports complete for horizons ending >=48h before the specified label cutoff; not guaranteed by input",
    "source_sha256": {
      "events.jsonl": "2ce178c72e6d4c2e95c46eeff010abe677dd667fb58881612ad414ad24afad98",
      "decision_times.jsonl": "6b101d088add1

## 1. Operational slices and probability reliability

Predefined slices: telemetry source, measurement age over 30 minutes versus fresher, and whether a trend is missing. Source is an audit attribute computed from available telemetry, not a model input. Show row and positive counts beside every metric. AP is omitted if a slice has only one class.

Reliability bins compare average predicted probability with observed incident fraction. Small bin counts and a single small test period do not establish reliable calibration. Brier score alone is not proof of calibration. No calibrator is fitted using these test outcomes.


In [2]:
scored = test.copy()
scored["probability"] = p
scored["baseline_probability"] = base_p
scored["freshness_slice"] = np.where(scored.measurement_age_minutes.isna(),"missing",np.where(scored.measurement_age_minutes>30,"older_than_30_minutes","within_30_minutes"))
scored["trend_slice"] = np.where(scored.temperature_trend_c_per_hour.isna(),"trend_missing","trend_present")
slice_rows = []
for dimension in ["source","freshness_slice","trend_slice"]:
    for value,group in scored.groupby(dimension,sort=True,dropna=False):
        slice_rows.append({"dimension":dimension,"slice":str(value),**wf.metrics(group.label,group.probability)})
slices = pd.DataFrame(slice_rows)
display(slices)
scored["probability_bin"] = pd.cut(scored.probability,[0,.05,.1,.2,.5,1],include_lowest=True)
reliability = scored.groupby("probability_bin",observed=True).agg(rows=("label","size"),positives=("label","sum"),mean_probability=("probability","mean"),observed_rate=("label","mean")).reset_index()
reliability["probability_bin"] = reliability["probability_bin"].astype(str)
display(reliability)


         dimension                  slice  ...  true_negative  false_negative
0           source         sensor-central  ...            100               0
1           source           sensor-coast  ...             88               1
2           source           sensor-north  ...             96               0
3  freshness_slice  older_than_30_minutes  ...            234               1
4  freshness_slice      within_30_minutes  ...             50               0
5      trend_slice          trend_missing  ...             68               0
6      trend_slice          trend_present  ...            216               1

[7 rows x 13 columns]
  probability_bin  rows  positives  mean_probability  observed_rate
0  (-0.001, 0.05]   274          1          0.002756        0.00365
1     (0.05, 0.1]     7          0          0.072226        0.00000
2      (0.1, 0.2]     4          0          0.132494        0.00000
3      (0.5, 1.0]    24         24          0.979671        1.00000


## 2. Approximate uncertainty with shipment resampling

Checkpoints from the same shipment can share outcomes. Resample entire shipments with replacement, not isolated rows. The following percentile ranges describe variation within this fixed test period; they do not capture all future drift or label incompleteness. Replicates without both classes cannot provide meaningful AP and are skipped for AP.


In [3]:
rng = np.random.default_rng(1729)
groups = [g for _,g in scored.groupby("shipment_id",sort=True)]
ap_values, brier_values = [], []
for _ in range(300):
    sample = pd.concat([groups[i] for i in rng.integers(0,len(groups),size=len(groups))],ignore_index=True)
    m = wf.metrics(sample.label,sample.probability)
    brier_values.append(m["brier"])
    if m["average_precision"] is not None:
        ap_values.append(m["average_precision"])
uncertainty = {"method":"300 shipment bootstrap replicates, seed 1729, percentile range; fixed-period approximation", "ap_valid_replicates":len(ap_values),
               "average_precision_95_range":np.quantile(ap_values,[.025,.975]).tolist() if ap_values else None,
               "brier_95_range":np.quantile(brier_values,[.025,.975]).tolist()}
result["uncertainty"] = uncertainty
print(uncertainty)


{'method': '300 shipment bootstrap replicates, seed 1729, percentile range; fixed-period approximation', 'ap_valid_replicates': 300, 'average_precision_95_range': [0.939207995951417, 1.0000000000000002], 'brier_95_range': [0.000285894710075891, 0.010140251368121817]}


## 3. Save the result and a notebook model artifact

Save the fitted pipeline, feature definitions, policy, versions, checksums, and reports. The pipeline includes imputation, missing indicators, scaling where applicable, and the model; new inputs must use the same feature order and definitions.

This artifact contains the notebook model. The engine uses the separate JSON artifact described in the runbook. Only load locally trusted joblib files: loading pickle-based formats can execute code. Matching checksums detect accidental changes, not authenticity. The next notebook checks loading in a fresh process and consistency checks before the production transition.


In [4]:
import joblib, hashlib, platform, sklearn, inspect
artifact = ROOT / 'personal' / 'outputs' / 'learning_model'
artifact.mkdir(parents=True,exist_ok=True)
joblib.dump(model,artifact / "model.joblib")
# Include exactly the feature interpretation used here, independent of notebook kernel state.
runtime_source = "from datetime import datetime, timedelta, timezone\nfrom statistics import mean\nimport math, json\n\n" + "\n\n".join(inspect.getsource(f) for f in [wf.utc,wf.known_revisions,wf.first_features,wf.window_features])
(artifact / "feature_runtime.py").write_text(runtime_source)
checksums = {name:hashlib.sha256((artifact/name).read_bytes()).hexdigest() for name in ["model.joblib","feature_runtime.py"]}
bundle = {"schema_version":1,"model_name":selection["selected_model"],"feature_order":wf.FEATURES,
          "lookback_hours":3,"reporting_grace_hours":48,"feature_clock_policy":"Exclude device_time later than received_at; latest by device time; lookback (left,right]", "model_fit_cutoff":selection["split"]["test_start"],"dependencies":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__,"joblib":joblib.__version__},"checksums":checksums,"split":selection["split"]}
bundle["model_version"] = hashlib.sha256(json.dumps(bundle,sort_keys=True).encode()).hexdigest()[:20]
wf.save_json(artifact / "manifest.json",bundle)
wf.save_json(ROOT / 'personal' / 'outputs' / 'notebook_results' / "final_evaluation.json",result)
for name,frame in [("checkpoint13_test_predictions",scored),("checkpoint13_test_slices",slices),("checkpoint13_reliability",reliability)]:
    frame.to_csv(ROOT / 'personal' / 'data' / 'tables' / (name+".csv"),index=False)
print("Saved fitted pipeline, feature runtime, version/policy manifest, and final evaluation.")
print("Test rows:",len(test),"positive outcomes:",int(test.label.sum()),"shipments:",test.shipment_id.nunique())


Saved fitted pipeline, feature runtime, version/policy manifest, and final evaluation.
Test rows: 309 positive outcomes: 25 shipments: 104


## 4. Limits and the next step

The held-out result concerns one synthetic period with assumed negative completeness, small positive counts, and engineered generator patterns. It cannot prove generalization to actual refrigerated fleets. No production threshold or launch decision is justified here.

Notebook 14 will use this selected artifact and the prior metrics to check serialization, unchanged predictions in a fresh Python process, deterministic feature reconstruction, and exact next production work. The engine methods, memory limits, concurrent reload and snapshots were implemented after this experiment. Their tests are separate from these model checks.

**Try explaining this:** why must we keep the validation-selected model even if the test result is disappointing?
